# Q2.7 Best Model Evaluation

Create an overlay plot of training vs test accuracy across all hyperparameter-search runs, identify high-train/low-test runs, and interpret the gap.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import wandb

ENTITY = 'anandhakrishnanm21-indian-institute-of-technology-madras'
SWEEP_PROJECT = 'q2_2_sweep_mnist'


In [ ]:
api = wandb.Api()
runs = api.runs(f'{ENTITY}/{SWEEP_PROJECT}')

rows = []
for r in runs:
    s = r.summary
    train_acc = s.get('train_acc')
    test_acc = s.get('test_acc')
    if train_acc is None or test_acc is None:
        continue
    rows.append({
        'run_name': r.name,
        'run_id': r.id,
        'train_acc': float(train_acc),
        'test_acc': float(test_acc),
        'gap': float(train_acc) - float(test_acc),
        'best_val_f1': float(s.get('best_val_f1')) if s.get('best_val_f1') is not None else np.nan,
    })

df = pd.DataFrame(rows)
if df.empty:
    raise RuntimeError('No runs with both train_acc and test_acc were found in the sweep project.')

df = df.sort_values('train_acc', ascending=False).reset_index(drop=True)
print(f'Loaded {len(df)} runs from {ENTITY}/{SWEEP_PROJECT}')
df.head(10)


In [ ]:
high_train_threshold = df['train_acc'].quantile(0.90)
large_gap_threshold = df['gap'].quantile(0.90)
overfit_df = df[(df['train_acc'] >= high_train_threshold) & (df['gap'] >= large_gap_threshold)].copy()

x = np.arange(len(df))
plt.figure(figsize=(12, 6))
plt.plot(x, df['train_acc'].values, label='Training Accuracy', linewidth=2)
plt.plot(x, df['test_acc'].values, label='Test Accuracy', linewidth=2)

if not overfit_df.empty:
    idx = overfit_df.index.values
    plt.scatter(idx, overfit_df['train_acc'].values, c='tab:red', s=45, label='High train + large gap', zorder=3)

plt.title('Q2.7: Training vs Test Accuracy Across Hyperparameter Search Runs')
plt.xlabel('Run Index (sorted by training accuracy)')
plt.ylabel('Accuracy')
plt.ylim(0, 1.05)
plt.grid(alpha=0.3)
plt.legend()
plt.show()

print(f'High-train threshold (90th pct): {high_train_threshold:.4f}')
print(f'Large-gap threshold (90th pct): {large_gap_threshold:.4f}')
print(f'Potential overfitting runs found: {len(overfit_df)}')
overfit_df[['run_name', 'run_id', 'train_acc', 'test_acc', 'gap', 'best_val_f1']].sort_values('gap', ascending=False).head(15)


### Interpretation
A large **train-test accuracy gap** means the model fit the training data much better than unseen data.

This indicates **overfitting** (poor generalization), typically caused by combinations like high model capacity, weak regularization, and/or overly aggressive training hyperparameters.
